# FLUX Beta: Unified Architecture & Demo Suite

This notebook verifies the final, unified Phase 8 FLUX model, repackages it into the modular `.flx` format (`Flux-beta.flx`), and runs a series of ultimate stress-test demos spanning all of FLUX's capabilities: byte-level processing, thermodynamic settling, episodic memory, and text generation.

In [ ]:
# ─────────────────────────────────────────────
# Cell 1: Environment Setup
# ─────────────────────────────────────────────
import os, sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    WORK_DIR = Path('/content/FLUX')
except ImportError:
    IN_COLAB = False
    WORK_DIR = Path.cwd()

if IN_COLAB:
    REPO_URL = "https://github.com/Unseengap/FLUX.git"
    if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
        subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=str(WORK_DIR))
        subprocess.run(['git', 'pull'], cwd=str(WORK_DIR))
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)])
    os.chdir(str(WORK_DIR))
    !pip install -q datasets rich faiss-cpu huggingface_hub matplotlib transformers scipy

sys.path.insert(0, str(WORK_DIR))
for i in range(1, 9):
    sys.path.insert(0, str(WORK_DIR / f'phases/phase{i}'))

OUTPUT_DIR = WORK_DIR / 'output' / 'flux_beta'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from flux_utils import PhaseLogger
log = PhaseLogger(phase=0)
log.separator("FLUX Beta: Unified Testing")
print(f"✓ Environment ready. Logs/Images will save to: {OUTPUT_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# Cell 2: Fetch Phase 8 from HF
# ─────────────────────────────────────────────
log.cell_start("Fetch Model")
import torch
from huggingface_hub import hf_hub_download
import shutil

ckpt_dir = WORK_DIR / 'checkpoints'
ckpt_dir.mkdir(exist_ok=True)
ckpt_path = ckpt_dir / 'phase8.phase.pt'

env_path = WORK_DIR / '.env'
if env_path.exists():
    for line in open(env_path):
        if line.startswith('hf_token='):
            os.environ['HF_TOKEN'] = line.strip().split('=', 1)[1]

if not ckpt_path.exists():
    print("Downloading from HuggingFace (UnseenGAP/FLUX)...")
    try:
        hf_path = hf_hub_download(repo_id="UnseenGAP/FLUX", filename="checkpoints/phase8.phase.pt")
        shutil.copy2(hf_path, ckpt_path)
        print("✓ Download successful")
    except Exception as e:
        print(f"⚠ Could not download: {e}")

log.cell_end("Fetch Model", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 3: Export to Modular Flux-beta.flx format
# ─────────────────────────────────────────────
log.cell_start("Export .flx")
from flux_large import FLUXLarge
from flux_utils import get_device
from datetime import datetime

print("Loading rigid Pytorch state dict...")
model = FLUXLarge.from_phase8_checkpoint(device=get_device())

flx_path = OUTPUT_DIR / 'Flux-beta.flx'
print(f"Repackaging into modular .flx format -> {flx_path.name}...")

flx_state = {
    'format': 'FLUX',
    'version': '1.0-beta',
    'metadata': {'compiled_on': datetime.now().isoformat()},
    
    'cse': {
        'config': {'wave_dim': 432},
        'state_dict': model.cse.state_dict(),
    },
    'field': {
        'config': model.config,
        'state_dict': model.field.state_dict(),
    },
    'memory': {
        'episodic': model.episodic_memory.save_state(),
        'working': model.working_memory.state_dict_memory(),
        'semantic': model.semantic_memory.save_state(),
    },
    'decoder': {
        # Strip prefix here as well just in case
        'state_dict': {k.replace('_orig_mod.', ''): v for k, v in model.decoder.state_dict().items()},
    },
    'bridges': {
        'wave_to_field': model.wave_to_field.state_dict(),
        'field_to_wave': model.field_to_wave.state_dict(),
    }
}

torch.save(flx_state, flx_path)
size_mb = flx_path.stat().st_size / 1e6
print(f"✓ Export complete: {size_mb:.1f} MB")
log.cell_end("Export .flx", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 4: Smoke Test — Reload from .flx
# ─────────────────────────────────────────────
log.cell_start("Smoke Test .flx")

del model # Destroy the old instance
import gc; gc.collect(); torch.cuda.empty_cache()

print("Loading model EXCLUSIVELY from .flx format...")
raw_flx = torch.load(flx_path, map_location='cpu', weights_only=False)
assert raw_flx['format'] == 'FLUX', "Not a valid FLUX file!"

re_model = FLUXLarge(config=raw_flx['field']['config'], device=get_device())

# Modular load
re_model.cse.load_state_dict(raw_flx['cse']['state_dict'])
re_model.field.load_state_dict(raw_flx['field']['state_dict'])
re_model.episodic_memory.load_state(raw_flx['memory']['episodic'])
re_model.decoder.load_state_dict(raw_flx['decoder']['state_dict'])
re_model.wave_to_field.load_state_dict(raw_flx['bridges']['wave_to_field'])

re_model = re_model.to(get_device())

stats = re_model.get_stats()
print(f"✓ FLUX-Beta Loaded: {stats.total_params:,} params")
print(f"✓ Episodic facts retained: {raw_flx['memory']['episodic'].get('count', 0)}")

assert stats.total_params > 60_000_000, "Model didn't load properly!"
log.cell_end("Smoke Test .flx", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 5: Demo A — Byte-Level Encoding (Phase 1/8)
# ─────────────────────────────────────────────
log.cell_start("Demo A: Byte Processing")
import matplotlib.pyplot as plt

print("Testing true byte-level processing (no Tokenizer OOV errors)...")
hard_prompts = [
    "Typycall misspelled wrds in englsh",
    "Emojis: 🧠⚡🤖",
    "Arabic: مرحباً بالعالم",
]

for p in hard_prompts:
    wave = re_model.cse.encode(p)
    print(f"  '{p}' -> Wave shape: {list(wave.full.shape)}")
    
# Generate from it
print("\nGenerations:")
for p in hard_prompts:
    out = re_model.generate(p, max_length=15)
    print(f"  {out}")
    
log.cell_end("Demo A", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 6: Demo B — Thermodynamic Settling Chart
# ─────────────────────────────────────────────
log.cell_start("Demo B: Thermodynamic Settling")

print("Visualizing field energy settling (Phase 4)...")

# We hook the TL module temporarily to grab energies
energies = []
orig_settle = re_model.tl.settle

def hooked_settle(field_state, wave_ctx):
    res = orig_settle(field_state, wave_ctx)
    energies.append(res['history'])
    return res

re_model.tl.settle = hooked_settle

re_model.forward("Quantum mechanics is governed by the Schrödinger equation.", learn=True)

re_model.tl.settle = orig_settle # restore

plt.figure(figsize=(8,4))
if energies and len(energies[0]) > 0:
    plt.plot(energies[0], marker='o', label="Energy (H)")
    plt.title("Thermodynamic Energy Minimization (Inference)")
    plt.xlabel("Settling Iteration")
    plt.ylabel("System Energy")
    plt.grid(True)
    plt_path = OUTPUT_DIR / 'thermodynamic_settling.png'
    plt.savefig(plt_path)
    print(f"✓ Chart saved to {plt_path}")
    plt.show()
else:
    print("Could not capture energy history.")

log.cell_end("Demo B", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 7: Demo C — Zero-Forgetting Verification
# ─────────────────────────────────────────────
log.cell_start("Demo C: Zero Forgetting")

print("Testing exactly 0.0% forgetting via Episodic Memory (Phase 6)...")
fact_A = "The secret base is located under Mount Shasta."
fact_B = "The capital of Earth in year 2500 is Neo-Tokyo."

print("\n--- Learning Fact A ---")
re_model.forward(fact_A, learn=True)
ans1, sim1 = re_model.query_memory("Where is the secret base?")
print(f"Recall A: '{ans1}' (Sim: {sim1:.3f})")

print("\n--- Learning 10 random facts to push memory... ---")
distractors = [
    "Apples are blue in dimension X.",
    "Gravity pulls upwards on Venus.",
    # ... Add a few to push index ...
    "Water is dry in the inverse universe.",
    "The password is 'FLUX'."
]
for d in distractors: re_model.forward(d, learn=True)

print("\n--- Learning Fact B ---")
re_model.forward(fact_B, learn=True)
ans2, sim2 = re_model.query_memory("What is the capital of Earth in 2500?")
print(f"Recall B: '{ans2}' (Sim: {sim2:.3f})")

print("\n--- Verifying Fact A is NOT overwritten ---")
ans3, sim3 = re_model.query_memory("Where is the secret base?")
print(f"Recall A again: '{ans3}' (Sim: {sim3:.3f})")

assert ans1 == ans3, "Model suffered catastrophic forgetting!"
print("\n✓ ZERO forgetting confirmed. Memory grew cleanly.")

log.cell_end("Demo C", "PASS")

In [ ]:
# ─────────────────────────────────────────────
# Cell 8: Save Final Run Logs to Drive
# ─────────────────────────────────────────────
log.cell_start("Wrap Up")
import shutil

if IN_COLAB:
    drive_out = Path('/content/drive/MyDrive/FLUX/output/flux_beta')
    drive_out.mkdir(parents=True, exist_ok=True)
    
    # Copy everything generated to Drive
    for file in OUTPUT_DIR.iterdir():
        if file.is_file():
            shutil.copy2(str(file), str(drive_out / file.name))
            print(f"> Pushed to Drive: {file.name}")
            
print("\n\nALL FLUX BETA TESTS COMPLETE.")
log.cell_end("Wrap Up", "PASS")